In [2]:
import pandas as pd
import numpy as np

In [3]:
columns = [
    "unit_number" if i == 0
    else "time_in_cycles" if i == 1
    else f"operational_setting_{i-1}" if i <= 4
    else f"sensor_measurement_{i-4}"
    for i in range(26)
]

In [4]:
columns_RUL = ["remaining_useful_life"]

In [5]:
raw_df_train = pd.read_csv("C:\\Users\\Badr\\Desktop\\Python_3.0\\CMAPSSData\\train_FD001.txt",sep=r'\s+',header=None,names=columns,index_col=False)
raw_df_test = pd.read_csv("C:\\Users\\Badr\\Desktop\\Python_3.0\\CMAPSSData\\test_FD001.txt",sep=r'\s+',header=None,names=columns,index_col=False)
raw_df_RUL = pd.read_csv("C:\\Users\\Badr\\Desktop\\Python_3.0\\CMAPSSData\\RUL_FD001.txt",sep=r'\s+',header=None,names=columns_RUL,index_col=False)

In [6]:
a = raw_df_RUL.index + 1
raw_df_RUL["unit_number"] = a
raw_df_RUL = raw_df_RUL[["unit_number","remaining_useful_life"]]

1)	unit number
2)	time, in cycles
3)	operational setting 1
4)	operational setting 2
5)	operational setting 3
6)	sensor measurement  1
7)	sensor measurement  2
...
26)	sensor measurement  21

In [7]:
raw_df_RUL.head()

,unit_number,remaining_useful_life
0,1,112
1,2,98
2,3,69
3,4,82
4,5,91


In [8]:
int_columns = ["unit_number","time_in_cycles"]
numeric_columns = columns.copy()
numeric_columns.remove("unit_number")
numeric_columns.remove("time_in_cycles")

In [9]:
for column in int_columns:
    raw_df_train[column] = raw_df_train[column].astype(int)
    raw_df_test[column] = raw_df_test[column].astype(int)
for column in numeric_columns:
    raw_df_train[column] = raw_df_train[column].astype(float)
    raw_df_test[column] = raw_df_test[column].astype(float)

## Data dictionary

This table documents the columns in the C-MAPSS FD001 dataset and explains their role in the predictive-maintenance project.

| Column name | Meaning | Unit | Data type | Category | Modeling role | Notes |
|---|---|---|---|---|---|---|
| `unit_number` | Identifier of the engine being observed | None | Integer | Identifier | Used for grouping and time-series splitting; not used directly as a sensor feature | Each engine has its own operating history |
| `time_in_cycles` | Operating age of the engine | Cycles | Integer | Time variable | Preserves sequence and may be used as a feature | Starts at 1 for each engine |
| `operational_setting_1` | Operating condition related to altitude | Dataset-specific numeric value | Float | Operating condition | Input feature | FD001 uses one main operating condition |
| `operational_setting_2` | Operating condition related to Mach number | Dataset-specific numeric value | Float | Operating condition | Input feature | Describes how the engine is being operated |
| `operational_setting_3` | Operating condition related to throttle resolver angle | Dataset-specific numeric value | Float | Operating condition | Input feature | Constant in FD001 but retained until a modeling decision is made |
| `sensor_measurement_1` | Total temperature at the fan inlet (T2) | Degrees Rankine (°R) | Float | Sensor | Candidate input feature | Constant in FD001 |
| `sensor_measurement_2` | Total temperature at the LPC outlet (T24) | Degrees Rankine (°R) | Float | Sensor | Candidate input feature | Sensor measurement |
| `sensor_measurement_3` | Total temperature at the HPC outlet (T30) | Degrees Rankine (°R) | Float | Sensor | Candidate input feature | Sensor measurement |
| `sensor_measurement_4` | Total temperature at the LPT outlet (T50) | Degrees Rankine (°R) | Float | Sensor | Candidate input feature | Sensor measurement |
| `sensor_measurement_5` | Pressure at the fan inlet (P2) | psia | Float | Sensor | Candidate input feature | Constant or nearly constant in FD001 |
| `sensor_measurement_6` | Total pressure in the bypass duct (P15) | psia | Float | Sensor | Candidate input feature | Constant or nearly constant in FD001 |
| `sensor_measurement_7` | Total pressure at the HPC outlet (P30) | psia | Float | Sensor | Candidate input feature | Sensor measurement |
| `sensor_measurement_8` | Physical fan speed (Nf) | rpm | Float | Sensor | Candidate input feature | Sensor measurement |
| `sensor_measurement_9` | Physical core speed (Nc) | rpm | Float | Sensor | Candidate input feature | Sensor measurement |
| `sensor_measurement_10` | Engine pressure ratio (epr) | Ratio | Float | Sensor | Candidate input feature | Constant in FD001 |
| `sensor_measurement_11` | Static pressure at the HPC outlet (Ps30) | psia | Float | Sensor | Candidate input feature | Sensor measurement |
| `sensor_measurement_12` | Ratio of fuel flow to Ps30 (phi) | pps/psi | Float | Sensor | Candidate input feature | Sensor measurement |
| `sensor_measurement_13` | Corrected fan speed (NRf) | rpm | Float | Sensor | Candidate input feature | Sensor measurement |
| `sensor_measurement_14` | Corrected core speed (NRc) | rpm | Float | Sensor | Candidate input feature | Sensor measurement |
| `sensor_measurement_15` | Bypass ratio (BPR) | Ratio | Float | Sensor | Candidate input feature | Sensor measurement |
| `sensor_measurement_16` | Burner fuel-air ratio (farB) | Ratio | Float | Sensor | Candidate input feature | Constant or nearly constant in FD001 |
| `sensor_measurement_17` | Bleed enthalpy (htBleed) | Dataset-specific numeric value | Float | Sensor | Candidate input feature | Sensor measurement |
| `sensor_measurement_18` | Demanded fan speed (Nf_dmd) | rpm | Float | Sensor | Candidate input feature | Constant in FD001 |
| `sensor_measurement_19` | Demanded corrected fan speed (PCNfR_dmd) | rpm | Float | Sensor | Candidate input feature | Constant in FD001 |
| `sensor_measurement_20` | HPT coolant bleed (W31) | lbm/s | Float | Sensor | Candidate input feature | Sensor measurement |
| `sensor_measurement_21` | LPT coolant bleed (W32) | lbm/s | Float | Sensor | Candidate input feature | Sensor measurement |
| `remaining_useful_life` | Number of operating cycles remaining before failure | Cycles | Integer | Target | Prediction target; never used as an input feature | Comes from the RUL file and is used for evaluation |

### Modeling rules

- `unit_number` is an identifier used to group observations by engine and prevent data leakage.
- `time_in_cycles` is the time variable used to preserve the order of observations.
- Operational settings and sensor measurements are candidate input features.
- `remaining_useful_life` is the target that the model must predict; it must not be given to the model as an input.
- Constant columns are documented first and will be removed only after a separate modeling decision.
- `estimated_failure_cycle` is a derived analysis column, not an original C-MAPSS column.
- The physical meanings and units come from the C-MAPSS technical documentation.


In [10]:
import os

from io import BytesIO
from datetime import datetime, timezone
import boto3
from botocore.client import Config
from dotenv import load_dotenv


load_dotenv()

BUCKET_NAME = os.environ["MINIO_BUCKET"]

s3_client = boto3.client(
    "s3",
    endpoint_url=os.environ["MINIO_ENDPOINT"],
    aws_access_key_id=os.environ["MINIO_ACCESS_KEY"],
    aws_secret_access_key=os.environ["MINIO_SECRET_KEY"],
    region_name="us-east-1",
    config=Config(signature_version="s3v4"),
)

pipeline_timestamp = datetime.now(timezone.utc)
pipeline_run_id = pipeline_timestamp.strftime("%Y%m%dT%H%M%S%fZ")
pipeline_load_date = pipeline_timestamp.strftime("%Y-%m-%d")
etl_metadata_records = []

In [11]:
if not raw_df_train.empty:

    parquet_buffer = BytesIO()

    raw_df_train.to_parquet(
        parquet_buffer,
        index=False,
        engine="fastparquet",
    )

    parquet_buffer.seek(0)
    object_key = (
    f"raw/train_data/"
    f"load_date={pipeline_load_date}/"
    f"run_id={pipeline_run_id}/"
    f"train.parquet"
    )

    s3_client.put_object(
        Bucket=BUCKET_NAME,
        Key=object_key,
        Body=parquet_buffer.getvalue(),
        ContentType="application/octet-stream",
    )

    print(f"Uploaded to MinIO: {object_key}")
    etl_metadata_records.append({
        "run_id": pipeline_run_id,
        "load_date": pipeline_load_date,
        "layer": "raw",
        "dataset_name": "train_data",
        "object_key": object_key,
        "status": "success",
        "row_count": len(raw_df_train),
        "recorded_at": datetime.now(timezone.utc).isoformat(),
    })

Uploaded to MinIO: raw/train_data/load_date=2026-09-02/run_id=20260902T120041719003Z/train.parquet


In [12]:
if not raw_df_test.empty:

    parquet_buffer = BytesIO()

    raw_df_test.to_parquet(
        parquet_buffer,
        index=False,
        engine="fastparquet",
    )

    parquet_buffer.seek(0)
    object_key = (
    f"raw/test_data/"
    f"load_date={pipeline_load_date}/"
    f"run_id={pipeline_run_id}/"
    f"test.parquet"
    )

    s3_client.put_object(
        Bucket=BUCKET_NAME,
        Key=object_key,
        Body=parquet_buffer.getvalue(),
        ContentType="application/octet-stream",
    )

    print(f"Uploaded to MinIO: {object_key}")
    etl_metadata_records.append({
        "run_id": pipeline_run_id,
        "load_date": pipeline_load_date,
        "layer": "raw",
        "dataset_name": "test_data",
        "object_key": object_key,
        "status": "success",
        "row_count": len(raw_df_test),
        "recorded_at": datetime.now(timezone.utc).isoformat(),
    })

Uploaded to MinIO: raw/test_data/load_date=2026-09-02/run_id=20260902T120041719003Z/test.parquet


In [13]:
if not raw_df_RUL.empty:

    parquet_buffer = BytesIO()

    raw_df_RUL.to_parquet(
        parquet_buffer,
        index=False,
        engine="fastparquet",
    )

    parquet_buffer.seek(0)
    object_key = (
    f"raw/RUL_data/"
    f"load_date={pipeline_load_date}/"
    f"run_id={pipeline_run_id}/"
    f"RUL.parquet"
    )

    s3_client.put_object(
        Bucket=BUCKET_NAME,
        Key=object_key,
        Body=parquet_buffer.getvalue(),
        ContentType="application/octet-stream",
    )

    print(f"Uploaded to MinIO: {object_key}")
    etl_metadata_records.append({
        "run_id": pipeline_run_id,
        "load_date": pipeline_load_date,
        "layer": "raw",
        "dataset_name": "RUL_data",
        "object_key": object_key,
        "status": "success",
        "row_count": len(raw_df_RUL),
        "recorded_at": datetime.now(timezone.utc).isoformat(),
    })

if len(etl_metadata_records) != 3:
    raise ValueError("Raw ingestion did not produce all three expected datasets.")

metadata_key = (
    f"metadata/etl_metadata/load_date={pipeline_load_date}/"
    f"run_id={pipeline_run_id}/manifest.parquet"
)
etl_metadata_df = pd.DataFrame(etl_metadata_records)
metadata_buffer = BytesIO()
etl_metadata_df.to_parquet(
    metadata_buffer,
    index=False,
    engine="fastparquet",
)
metadata_buffer.seek(0)
s3_client.put_object(
    Bucket=BUCKET_NAME,
    Key=metadata_key,
    Body=metadata_buffer.getvalue(),
    ContentType="application/octet-stream",
)
print(f"Uploaded ETL metadata manifest: {metadata_key}")

Uploaded to MinIO: raw/RUL_data/load_date=2026-09-02/run_id=20260902T120041719003Z/RUL.parquet
Uploaded ETL metadata manifest: metadata/etl_metadata/load_date=2026-09-02/run_id=20260902T120041719003Z/manifest.parquet
